In [ ]:
# ============================================================
# Cell 0 — Mount Drive, install libraries, define paths
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!nvidia-smi || echo "No NVIDIA GPU found. Change runtime to GPU for speed."

# Install libs (pinned similar to your previous runs)
!pip -q install \
  "supervision>=0.19.0" \
  opencv-python-headless==4.10.0.84 \
  "huggingface_hub>=0.34,<1.0" \
  "tqdm>=4.66.0" \
  "transformers>=4.40" "accelerate>=0.25" "timm>=0.9.12" pillow scikit-learn \
  "segment-anything==1.0"

from pathlib import Path

# ---- ROOTS ----
DRIVE_ROOT = Path("/content/drive/MyDrive")
GDINO_ROOT = DRIVE_ROOT / "pavex_bbox_experiments"

# Same raw dataset as before
BIG_ROOT      = DRIVE_ROOT / "BigDataset"
IMG_DIR       = BIG_ROOT / "augmented"      # augmented images
GT_LABELS_DIR = BIG_ROOT / "labels_gt"   # 200 YOLOv8-seg txt (GT masks)

# New run root so you can compare later
RUN_NAME  = "runs_sam_refined_v11b_augmented"
RUN_ROOT  = BIG_ROOT / RUN_NAME

PRED_SEG_DIR      = RUN_ROOT / "labels_pred_seg"
OVERLAY_ALL_DIR   = RUN_ROOT / "overlays_all_seg"
OVERLAY_GT_DIR    = RUN_ROOT / "overlays_seg_eval"
EVAL_DIR          = RUN_ROOT / "eval"

for d in [PRED_SEG_DIR, OVERLAY_ALL_DIR, OVERLAY_GT_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("IMG_DIR        :", IMG_DIR)
print("GT_LABELS_DIR  :", GT_LABELS_DIR)
print("RUN_ROOT       :", RUN_ROOT)
print("PRED_SEG_DIR   :", PRED_SEG_DIR)
print("OVERLAY_ALL_DIR:", OVERLAY_ALL_DIR)
print("OVERLAY_GT_DIR :", OVERLAY_GT_DIR)


Mounted at /content/drive
Thu Nov 27 08:58:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             30W /   70W |    9958MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
# ============================================================
# Cell 1 — Unzip v11b pack + read pipeline_config & seed_input
# ============================================================
import json

EXPORTS   = GDINO_ROOT / "exports"
PACK_ZIP  = EXPORTS / "pavex_sam_seed_pack_v11b.zip"
PACK_ROOT = EXPORTS / "pavex_sam_seed_pack_v11b"

if PACK_ZIP.exists() and not PACK_ROOT.exists():
    print("Unzipping v11b pack...")
    !unzip -oq "{PACK_ZIP}" -d "{EXPORTS}"

PIPELINE_CFG = PACK_ROOT / "pipeline_config.json"
SEED_INPUT   = PACK_ROOT / "seed_input.json"

if not PIPELINE_CFG.exists():
    raise FileNotFoundError(f"pipeline_config.json not found at {PIPELINE_CFG}")

with open(PIPELINE_CFG, "r") as f:
    pipe = json.load(f)

# ---- BASIC SETTINGS FROM PACK ----
paths_info = pipe.get("paths", {})
HF_REPO    = paths_info.get("hf_repo_grounding_dino", "IDEA-Research/grounding-dino-base")
sam_fname  = paths_info.get("sam_weights_file", None)

models_info = pipe.get("models", {})
sam_info    = models_info.get("SAM", {})
SAM_ARCH    = sam_info.get("arch", None)

PROMPTS_MP = pipe.get("prompts", ["pothole", "road hole", "damaged road"])

thr = pipe.get("thresholds", {})
TEXT_THR = float(thr.get("TEXT_THR", 0.25))
BOX_THR  = float(thr.get("BOX_THR",  0.35))
NMS_THR  = float(thr.get("NMS_THR",  0.45))

# ---- POST-FILTERS (mask area, solidity, edge rule) ----
post_filters = pipe.get("post_filters", {})

MAX_AREA_FRAC      = float(post_filters.get("max_area_frac", 0.65))
edge_cfg           = post_filters.get("edge_touches_rule", {})
EDGE_RULE_ON       = bool(edge_cfg.get("on", False))
EDGE_AREA_FRAC_GATE = float(edge_cfg.get("area_frac_gate", 0.40))
EDGE_MIN_EDGES_TOUCHED = int(edge_cfg.get("min_edges_touched", 3))

# From seed_input.json (min_area, solidity_min)
MIN_MASK_AREA   = 200.0
SOLIDITY_MIN    = 0.48
if SEED_INPUT.exists():
    with open(SEED_INPUT, "r") as f:
        seed_in = json.load(f)
    pf_seed = (seed_in.get("POST_FILTERS") or {})
    MIN_MASK_AREA = float(pf_seed.get("min_area", MIN_MASK_AREA))
    MAX_AREA_FRAC = float(pf_seed.get("max_area_frac", MAX_AREA_FRAC))
    SOLIDITY_MIN  = float(pf_seed.get("solidity_min", SOLIDITY_MIN))

# SAM weights path (inside pack/models)
SAM_WEIGHTS_PATH = None
if sam_fname:
    candidate = PACK_ROOT / "models" / sam_fname
    if candidate.exists():
        SAM_WEIGHTS_PATH = candidate

print("HF_REPO          :", HF_REPO)
print("PROMPTS_MP       :", PROMPTS_MP)
print("TEXT_THR, BOX_THR, NMS_THR:", TEXT_THR, BOX_THR, NMS_THR)
print("SAM_ARCH         :", SAM_ARCH)
print("SAM_WEIGHTS_PATH :", SAM_WEIGHTS_PATH)
print("--- Post-filters ---")
print("MIN_MASK_AREA    :", MIN_MASK_AREA)
print("MAX_AREA_FRAC    :", MAX_AREA_FRAC)
print("SOLIDITY_MIN     :", SOLIDITY_MIN)
print("EDGE_RULE_ON     :", EDGE_RULE_ON)
print("EDGE_AREA_FRAC_GATE  :", EDGE_AREA_FRAC_GATE)
print("EDGE_MIN_EDGES_TOUCHED:", EDGE_MIN_EDGES_TOUCHED)


HF_REPO          : IDEA-Research/grounding-dino-base
PROMPTS_MP       : ['potholes.', 'pothole.', 'road pothole.', 'asphalt hole.']
TEXT_THR, BOX_THR, NMS_THR: 0.4 0.24 0.33
SAM_ARCH         : vit_h
SAM_WEIGHTS_PATH : /content/drive/MyDrive/pavex_bbox_experiments/exports/pavex_sam_seed_pack_v11b/models/sam_vit_h_4b8939.pth
--- Post-filters ---
MIN_MASK_AREA    : 200.0
MAX_AREA_FRAC    : 0.5
SOLIDITY_MIN     : 0.6
EDGE_RULE_ON     : True
EDGE_AREA_FRAC_GATE  : 0.4
EDGE_MIN_EDGES_TOUCHED: 3


In [ ]:
# ============================================================
# Cell 2 — Load GroundingDINO & SAM
# ============================================================
import torch
from transformers import AutoProcessor, GroundingDinoForObjectDetection
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# GroundingDINO
processor = AutoProcessor.from_pretrained(HF_REPO)
gdino_model = GroundingDinoForObjectDetection.from_pretrained(HF_REPO)
gdino_model.to(device)
gdino_model.eval()
print("GDINO model device:", next(gdino_model.parameters()).device)

# SAM
sam_predictor = None
if SAM_ARCH is not None and SAM_WEIGHTS_PATH is not None and SAM_WEIGHTS_PATH.exists():
    print("Loading SAM:", SAM_ARCH, "from", SAM_WEIGHTS_PATH)
    sam_model = sam_model_registry[SAM_ARCH](checkpoint=str(SAM_WEIGHTS_PATH))
    sam_model.to(device)
    sam_predictor = SamPredictor(sam_model)
    print("SAM loaded on", device)
else:
    print("WARNING: SAM not loaded. Segmentation will not work.")


Using device: cuda
GDINO model device: cuda:0
Loading SAM: vit_h from /content/drive/MyDrive/pavex_bbox_experiments/exports/pavex_sam_seed_pack_v11b/models/sam_vit_h_4b8939.pth
SAM loaded on cuda


In [ ]:
# ============================================================
# Cell 3 — Generic utilities: IoU, overlays, masks, GT reader
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

# ------------ BOX IOU / NMS ------------
def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    if union <= 0:
        return 0.0
    return inter / union

def nms_boxes(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return boxes, scores
    boxes = boxes.copy()
    scores = scores.copy()
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([iou_xyxy(boxes[i], boxes[j]) for j in rest])
        rest = rest[ious < iou_thr]
        order = rest
    keep = np.array(keep, dtype=int)
    return boxes[keep], scores[keep]

# ------------ LABEL PATH → IMAGE PATH (Roboflow-style) ------------
def label_to_image_path(label_path: Path):
    """
    Map Roboflow-style labels to image filenames:
      img_0002_jpg.rf.xxxxx.txt  -> images/img_0002.jpg (or .png/.jpeg)
    """
    stem = label_path.stem
    img_stem = stem
    for marker in ["_jpg.rf.", "_png.rf.", "_jpeg.rf."]:
        if marker in stem:
            img_stem = stem.split(marker)[0]
            break

    candidates = [
        IMG_DIR / f"{img_stem}.jpg",
        IMG_DIR / f"{img_stem}.jpeg",
        IMG_DIR / f"{img_stem}.png",
        IMG_DIR / f"{img_stem}.JPG",
        IMG_DIR / f"{img_stem}.PNG",
    ]
    for p in candidates:
        if p.exists():
            return p

    print(f"[WARN] No image found matching label {label_path.name}")
    return None

# ------------ OVERLAYS ------------
def save_overlay_boxes_and_masks(
    img_path: Path,
    boxes_xyxy,
    masks,
    out_dir: Path,
):
    """
    Draw predicted masks (transparent red) + boxes (green).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(img_path))
    if img is None:
        return
    overlay = img.copy()

    # masks
    if masks is not None:
        for m in masks:
            m_bin = (m > 0).astype(np.uint8)
            if m_bin.sum() == 0:
                continue
            color = np.zeros_like(img)
            color[:, :, 2] = 255  # red
            mask_3 = m_bin[..., None]
            overlay = np.where(
                mask_3 == 1,
                (0.2 * color + 0.8 * overlay).astype(overlay.dtype),
                overlay,
            )

    # boxes
    if boxes_xyxy is not None and len(boxes_xyxy) > 0:
        for (x1, y1, x2, y2) in boxes_xyxy.astype(int):
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

    out_path = out_dir / img_path.name
    cv2.imwrite(str(out_path), overlay)

# ------------ MASK METRICS ------------
def mask_iou(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    if union == 0:
        return 0.0
    return inter / union

def mask_dice(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    s1 = m1.sum()
    s2 = m2.sum()
    denom = s1 + s2
    if denom == 0:
        return 0.0
    return 2.0 * inter / denom

# ------------ MASK NMS ------------
def mask_nms(masks, scores, iou_thr=0.7):
    if len(masks) == 0:
        return [], np.array([], dtype=float)

    masks = list(masks)
    scores = np.array(scores, dtype=float)
    order = scores.argsort()[::-1]
    keep_indices = []
    used = np.zeros(len(masks), dtype=bool)

    while len(order) > 0:
        i = order[0]
        keep_indices.append(i)
        used[i] = True
        keep_rest = []
        for j in order[1:]:
            if used[j]:
                continue
            if mask_iou(masks[i], masks[j]) <= iou_thr:
                keep_rest.append(j)
        order = np.array(keep_rest, dtype=int)

    masks_kept = [masks[i] for i in keep_indices]
    scores_kept = scores[keep_indices]
    return masks_kept, scores_kept

def suppress_nested_masks_by_area(masks, scores,
                                  iou_thr=0.7,
                                  area_ratio_thr=0.7):
    """
    Drop 'child' masks that lie mostly inside a much larger 'mother' mask.
    """
    n = len(masks)
    if n <= 1:
        return masks, scores

    masks = list(masks)
    scores = np.array(scores, dtype=float)
    areas = np.array([(m > 0).sum() for m in masks], dtype=float)
    keep = np.ones(n, dtype=bool)

    for i in range(n):
        if not keep[i]:
            continue
        for j in range(n):
            if i == j or not keep[j]:
                continue
            iou_ij = mask_iou(masks[i], masks[j])
            if iou_ij < iou_thr:
                continue
            if areas[i] >= areas[j]:
                larger, smaller = i, j
            else:
                larger, smaller = j, i
            ratio = areas[smaller] / max(areas[larger], 1e-6)
            if ratio <= area_ratio_thr:
                keep[smaller] = False

    masks_kept  = [m for m, k in zip(masks, keep) if k]
    scores_kept = scores[keep]
    return masks_kept, scores_kept

# ------------ YOLOv8-seg GT reader ------------
def read_yolo_seg_masks(label_path: Path, img_w: int, img_h: int):
    if not label_path.exists():
        return []
    text = label_path.read_text().strip()
    if not text:
        return []

    h, w = img_h, img_w
    masks = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 6:
            continue
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue
        xs = coords[0::2]
        ys = coords[1::2]
        pts = []
        for x_norm, y_norm in zip(xs, ys):
            x = int(x_norm * w)
            y = int(y_norm * h)
            pts.append([x, y])
        pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 255)
        masks.append(mask)

    return masks


In [ ]:
# ============================================================
# Cell 4 — GDINO predictor, SAM masks, mask filters
# ============================================================
def gdino_predict_for_image(
    img_path: Path,
    box_thr: float = BOX_THR,
    text_thr: float = TEXT_THR,
    nms_thr: float = NMS_THR,
    margin_frac: float = 0.02,
    min_road_area_frac: float = 0.20,
    min_road_height_frac: float = 0.30,
    min_road_width_frac: float = 0.30,
):
    """
    Run GroundingDINO on one image and return filtered boxes.

    - Uses prompts from PROMPTS_MP.
    - Applies a simple road-band filter to remove giant bands across edges.
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        raise ValueError(f"Failed to read image: {img_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    img_area = float(w * h)

    prompt_str = " . ".join(PROMPTS_MP)
    inputs = processor(
        images=img_rgb,
        text=prompt_str,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        threshold=float(box_thr),
        target_sizes=[(h, w)]
    )[0]

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    widths  = np.clip(x2 - x1, 0, None)
    heights = np.clip(y2 - y1, 0, None)
    areas   = widths * heights

    area_fracs   = areas / img_area
    height_fracs = heights / float(h)
    width_fracs  = widths  / float(w)

    # valid content edges (ignore small grey border)
    margin_x = int(margin_frac * w)
    margin_y = int(margin_frac * h)

    left_edge_valid   = margin_x
    right_edge_valid  = w - margin_x
    top_edge_valid    = margin_y
    bottom_edge_valid = h - margin_y

    touches_left   = x1 <= left_edge_valid
    touches_right  = x2 >= right_edge_valid
    touches_top    = y1 <= top_edge_valid
    touches_bottom = y2 >= bottom_edge_valid

    horiz_road = (
        touches_left & touches_right &
        (height_fracs >= min_road_height_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    vert_road = (
        touches_top & touches_bottom &
        (width_fracs  >= min_road_width_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    huge_box = area_fracs >= 0.75

    road_like = horiz_road | vert_road | huge_box

    keep_mask = ~road_like
    boxes  = boxes[keep_mask]
    scores = scores[keep_mask]

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    boxes_nms, scores_nms = nms_boxes(boxes, scores, iou_thr=nms_thr)
    return boxes_nms, scores_nms, (w, h)


def sam_masks_from_boxes(img_bgr, boxes_xyxy, scores=None):
    """
    Run SAM for each GDINO box and return binary masks (0/1).
    """
    if sam_predictor is None or boxes_xyxy is None or len(boxes_xyxy) == 0:
        return [], np.array([], dtype=float)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    sam_predictor.set_image(img_rgb)

    if scores is None:
        scores = np.ones(len(boxes_xyxy), dtype=float)

    masks_out = []
    scores_out = []

    for box, score in zip(boxes_xyxy, scores):
        box_arr = np.array(box, dtype=np.float32)[None, :]
        with torch.no_grad():
            mask, _, _ = sam_predictor.predict(
                box=box_arr,
                point_coords=None,
                point_labels=None,
                multimask_output=False
            )
        m = mask[0]
        if m is None or m.size == 0:
            continue
        m_bin = (m > 0).astype(np.uint8)
        if m_bin.sum() == 0:
            continue
        masks_out.append(m_bin)
        scores_out.append(float(score))

    if not masks_out:
        return [], np.array([], dtype=float)

    return masks_out, np.array(scores_out, dtype=float)


def mask_solidity(mask: np.ndarray) -> float:
    """
    Approximate solidity = area / convex_hull_area.
    """
    m = (mask > 0).astype(np.uint8)
    if m.sum() < 5:
        return 0.0
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    if area <= 0:
        return 0.0
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    if hull_area <= 0:
        return 0.0
    return float(area / hull_area)


def filter_masks_with_postfilters(masks, boxes_xyxy, scores,
                                  min_area: float = MIN_MASK_AREA,
                                  max_area_frac: float = MAX_AREA_FRAC,
                                  solidity_min: float = SOLIDITY_MIN,
                                  edge_rule_on: bool = EDGE_RULE_ON,
                                  edge_area_frac_gate: float = EDGE_AREA_FRAC_GATE,
                                  edge_min_edges_touched: int = EDGE_MIN_EDGES_TOUCHED):
    """
    Apply area / solidity / edge-touch filters using config from pack.
    Returns filtered (masks, boxes, scores).
    """
    if len(masks) == 0:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    h, w = masks[0].shape[:2]
    img_area = float(w * h)

    kept_masks = []
    kept_boxes = []
    kept_scores = []

    for m, box, s in zip(masks, boxes_xyxy, scores):
        area = (m > 0).sum()
        if area < min_area:
            continue

        area_frac = area / img_area
        if area_frac > max_area_frac:
            continue

        sol = mask_solidity(m)
        if sol < solidity_min:
            continue

        # optional edge-touch rule (road-like giant region)
        if edge_rule_on:
            x1, y1, x2, y2 = box
            touches_top = y1 <= 0.01 * h
            touches_bottom = y2 >= 0.99 * h
            touches_left = x1 <= 0.01 * w
            touches_right = x2 >= 0.99 * w
            edges_touched = sum([touches_top, touches_bottom, touches_left, touches_right])
            if area_frac >= edge_area_frac_gate and edges_touched >= edge_min_edges_touched:
                # likely huge border region
                continue

        kept_masks.append(m)
        kept_boxes.append(box)
        kept_scores.append(s)

    if not kept_masks:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    return kept_masks, np.vstack(kept_boxes), np.array(kept_scores, dtype=float)


In [ ]:
# ============================================================
# Cell 5 — Binary mask → YOLOv8-seg label + auto-annotate all
# ============================================================
def binary_mask_to_yolov8_seg_line(
    mask,
    img_w: int,
    img_h: int,
    cls_id: int = 0,
    simplify_eps: float = 1.5,
):
    """
    Convert a binary mask (H,W) into one YOLOv8-seg label line:
      cls cx cy w h x1 y1 x2 y2 ...
    """
    if mask.dtype != np.uint8:
        m = (mask > 0).astype(np.uint8)
    else:
        m = (mask > 0).astype(np.uint8)

    if m.sum() == 0:
        return None

    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < 10:
        return None

    if simplify_eps > 0:
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, simplify_eps, True)
        if len(approx) >= 3:
            cnt = approx

    pts = cnt.reshape(-1, 2).astype(float)

    x_min = pts[:, 0].min()
    y_min = pts[:, 1].min()
    x_max = pts[:, 0].max()
    y_max = pts[:, 1].max()

    w_box = x_max - x_min
    h_box = y_max - y_min
    if w_box <= 0 or h_box <= 0:
        return None

    cx = x_min + w_box / 2.0
    cy = y_min + h_box / 2.0

    cx_n = cx / img_w
    cy_n = cy / img_h
    w_n  = w_box / img_w
    h_n  = h_box / img_h

    poly_norm = []
    for (x, y) in pts:
        poly_norm.append(x / img_w)
        poly_norm.append(y / img_h)

    parts = [str(cls_id), f"{cx_n:.6f}", f"{cy_n:.6f}", f"{w_n:.6f}", f"{h_n:.6f}"]
    parts += [f"{v:.6f}" for v in poly_norm]
    return " ".join(parts)


def auto_annotate_all_images():
    # collect all images
    img_paths = sorted(
        list(IMG_DIR.glob("*.jpg")) +
        list(IMG_DIR.glob("*.jpeg")) +
        list(IMG_DIR.glob("*.png")) +
        list(IMG_DIR.glob("*.JPG")) +
        list(IMG_DIR.glob("*.PNG"))
    )
    print(f"Found {len(img_paths)} images in {IMG_DIR}")

    for img_path in tqdm(img_paths, desc="Auto-annotating all images"):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"[WARN] Failed to read {img_path}")
            continue

        h, w = img_bgr.shape[:2]

        # 1) GDINO boxes
        boxes_pred, scores_pred, (img_w, img_h) = gdino_predict_for_image(img_path)
        if img_w != w or img_h != h:
            w, h = img_w, img_h

        # 2) SAM masks
        masks_pred, mask_scores = sam_masks_from_boxes(img_bgr, boxes_pred, scores_pred)

        # 3) NMS + nested suppression
        masks_pred, mask_scores = mask_nms(masks_pred, mask_scores, iou_thr=0.7)
        masks_pred, mask_scores = suppress_nested_masks_by_area(
            masks_pred, mask_scores, iou_thr=0.7, area_ratio_thr=0.7
        )

        # 4) Post-filters (area, solidity, edges)
        masks_pred, boxes_pred, mask_scores = filter_masks_with_postfilters(
            masks_pred, boxes_pred, mask_scores
        )

        # 5) Overlays for QA
        save_overlay_boxes_and_masks(img_path, boxes_pred, masks_pred, out_dir=OVERLAY_ALL_DIR)

        # 6) YOLOv8-seg labels
        out_txt = PRED_SEG_DIR / f"{img_path.stem}.txt"
        if not masks_pred:
            out_txt.write_text("")
            continue

        lines = []
        for m in masks_pred:
            line = binary_mask_to_yolov8_seg_line(m, img_w=w, img_h=h, cls_id=0, simplify_eps=1.5)
            if line is not None:
                lines.append(line)

        if lines:
            out_txt.write_text("\n".join(lines) + "\n")
        else:
            out_txt.write_text("")

    print("Done auto-annotate.")
    print("Labels in :", PRED_SEG_DIR)
    print("Overlays in:", OVERLAY_ALL_DIR)


# Run auto-annotation
auto_annotate_all_images()


Found 11818 images in /content/drive/MyDrive/BigDataset/augmented


Auto-annotating all images: 100%|██████████| 11818/11818 [11:19:15<00:00,  3.45s/it]

Done auto-annotate.
Labels in : /content/drive/MyDrive/BigDataset/runs_sam_refined_v11b_augmented/labels_pred_seg
Overlays in: /content/drive/MyDrive/BigDataset/runs_sam_refined_v11b_augmented/overlays_all_seg


In [ ]:
# ============================================================
# Cell 6 — Segmentation-based evaluation on 200 GT images
# ============================================================
import pandas as pd
import json as _json_mod

IOU_THR_EVAL = 0.5

def eval_dataset_on_gt_seg():
    if sam_predictor is None:
        print("SAM is not loaded; cannot run segmentation-based evaluation.")
        return None, None

    gt_label_paths = sorted(GT_LABELS_DIR.glob("*.txt"))
    print(f"Found {len(gt_label_paths)} GT label files")

    per_img_stats = []
    skipped_labels = []

    total_TP = total_FP = total_FN = 0
    neg_imgs = 0
    total_FP_neg = 0
    ious_all = []
    dices_all = []

    for lbl_path in tqdm(gt_label_paths, desc="Evaluating SEG on GT subset"):
        img_path = label_to_image_path(lbl_path)
        if img_path is None:
            skipped_labels.append(lbl_path.name)
            continue

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"[WARN] Cannot read image {img_path}")
            skipped_labels.append(lbl_path.name)
            continue
        h, w = img_bgr.shape[:2]

        # Predictions: GDINO + SAM + post-filters
        boxes_pred, scores_pred, (img_w, img_h) = gdino_predict_for_image(img_path)
        assert img_w == w and img_h == h

        masks_pred, mask_scores = sam_masks_from_boxes(img_bgr, boxes_pred, scores_pred)
        masks_pred, mask_scores = mask_nms(masks_pred, mask_scores, iou_thr=0.7)
        masks_pred, mask_scores = suppress_nested_masks_by_area(
            masks_pred, mask_scores, iou_thr=0.7, area_ratio_thr=0.7
        )
        masks_pred, boxes_pred, mask_scores = filter_masks_with_postfilters(
            masks_pred, boxes_pred, mask_scores
        )
        n_pred = len(masks_pred)

        # GT masks
        masks_gt = read_yolo_seg_masks(lbl_path, img_w, img_h)
        n_gt = len(masks_gt)

        # Overlays for QA on GT subset
        save_overlay_boxes_and_masks(img_path, boxes_pred, masks_pred, out_dir=OVERLAY_GT_DIR)

        # Negative GT image
        if n_gt == 0:
            neg_imgs += 1
            total_FP_neg += n_pred
            total_FP += n_pred
            per_img_stats.append({
                "image": img_path.name,
                "n_gt": n_gt,
                "n_pred": n_pred,
                "TP": 0,
                "FP": n_pred,
                "FN": 0,
                "IoU_mean": float("nan"),
                "Dice_mean": float("nan"),
            })
            continue

        # GT has masks but predictions none
        if n_pred == 0:
            total_FN += n_gt
            per_img_stats.append({
                "image": img_path.name,
                "n_gt": n_gt,
                "n_pred": n_pred,
                "TP": 0,
                "FP": 0,
                "FN": n_gt,
                "IoU_mean": float("nan"),
                "Dice_mean": float("nan"),
            })
            continue

        # IoU & Dice matrix
        iou_mat = np.zeros((n_gt, n_pred), dtype=float)
        dice_mat = np.zeros((n_gt, n_pred), dtype=float)
        for i in range(n_gt):
            for j in range(n_pred):
                iou_val = mask_iou(masks_gt[i], masks_pred[j])
                dice_val = mask_dice(masks_gt[i], masks_pred[j])
                iou_mat[i, j] = iou_val
                dice_mat[i, j] = dice_val

        matched_gt = set()
        matched_pred = set()
        tps = 0
        ious_img = []
        dices_img = []

        pairs = []
        for i in range(n_gt):
            for j in range(n_pred):
                pairs.append((iou_mat[i, j], i, j))
        pairs.sort(reverse=True, key=lambda x: x[0])

        for iou_val, i, j in pairs:
            if iou_val < IOU_THR_EVAL:
                break
            if i in matched_gt or j in matched_pred:
                continue
            matched_gt.add(i)
            matched_pred.add(j)
            tps += 1
            ious_img.append(iou_val)
            dices_img.append(dice_mat[i, j])

        fps = n_pred - tps
        fns = n_gt - tps

        total_TP += tps
        total_FP += fps
        total_FN += fns

        if ious_img:
            ious_all.extend(ious_img)
            dices_all.extend(dices_img)
            iou_mean_img = float(np.mean(ious_img))
            dice_mean_img = float(np.mean(dices_img))
        else:
            iou_mean_img = float("nan")
            dice_mean_img = float("nan")

        per_img_stats.append({
            "image": img_path.name,
            "n_gt": n_gt,
            "n_pred": n_pred,
            "TP": tps,
            "FP": fps,
            "FN": fns,
            "IoU_mean": iou_mean_img,
            "Dice_mean": dice_mean_img,
        })

    precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0.0
    recall    = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0.0
    iou_mean_dataset  = float(np.mean(ious_all))  if ious_all else float("nan")
    dice_mean_dataset = float(np.mean(dices_all)) if dices_all else float("nan")
    fp_per_neg_img = total_FP_neg / neg_imgs if neg_imgs > 0 else float("nan")

    summary = {
        "TP": total_TP,
        "FP": total_FP,
        "FN": total_FN,
        "precision": precision,
        "recall": recall,
        "IoU_mean": iou_mean_dataset,
        "Dice_mean": dice_mean_dataset,
        "neg_imgs": neg_imgs,
        "FP_per_neg_img": fp_per_neg_img,
        "skipped_labels": skipped_labels,
    }

    # Save to disk (for comparison across runs)
    df = pd.DataFrame(per_img_stats)
    df.to_csv(EVAL_DIR / "seg_eval_per_image.csv", index=False)
    with open(EVAL_DIR / "seg_eval_summary.json", "w") as f:
        _json_mod.dump(summary, f, indent=2)

    print("Saved per-image stats  ->", EVAL_DIR / "seg_eval_per_image.csv")
    print("Saved summary JSON     ->", EVAL_DIR / "seg_eval_summary.json")
    return per_img_stats, summary


per_img_stats_seg, summary_seg = eval_dataset_on_gt_seg()
summary_seg


Found 200 GT label files


Evaluating SEG on GT subset: 100%|██████████| 200/200 [11:31<00:00,  3.46s/it]

Saved per-image stats  -> /content/drive/MyDrive/BigDataset/runs_sam_refined_v11b/eval/seg_eval_per_image.csv
Saved summary JSON     -> /content/drive/MyDrive/BigDataset/runs_sam_refined_v11b/eval/seg_eval_summary.json


{'TP': 241,
 'FP': 131,
 'FN': 24,
 'precision': 0.6478494623655914,
 'recall': 0.909433962264151,
 'IoU_mean': 0.8379909303598456,
 'Dice_mean': 0.9070538535078647,
 'neg_imgs': 0,
 'FP_per_neg_img': nan,
 'skipped_labels': []}